In [18]:
import pandas as pd
file_path = "../data/processed/clean_book_summaries.csv"
df = pd.read_csv(file_path)
df.sample(5)

,Title,Author,Genres,Summary
12632,Dreams of the Raven,NaN,['Science Fiction'],A mysterious distress call leads to the USS E...
1598,Distress,Greg Egan,"['Science Fiction', 'Speculative fiction', 'Fa...",It describes the political intrigue surroundi...
10002,Dead Right,Peter Robinson,"['Crime Fiction', 'Mystery']","On a rainy night in Eastvale, a teenager is f..."
1415,The Nutmeg of Consolation,Patrick O'Brian,"['Historical fiction', 'Fiction', 'Historical ...",The Nutmeg of Consolation opens with Aubrey a...
11376,A Mercy,Toni Morrison,['Novel'],"Florens, a slave, lives and works on Jacob Va..."


In [19]:
for col in ["Tone", "Pacing", "Aesthetic", "Themes"]:
    if col not in df.columns:
        df[col] = None

In [20]:
df.head()

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca...",None,None,None,None
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan...",None,None,None,None
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...,None,None,None,None
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,None,None,None,None
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...,None,None,None,None


In [21]:
import os
from dotenv import load_dotenv
from groq import Groq
load_dotenv("../../.env")

True

In [22]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [34]:
promptCh = """You are a STRICT semantic classification engine.

Your task is to classify a book using ONLY the predefined labels below.

You will receive:
- Title (supporting signal)
- Existing genres (noisy hints, optional)
- Book summary (PRIMARY source of truth)

CLASSIFICATION RULES:

1) The BOOK SUMMARY is the primary source. The title may help to get faster meaning.
2) Existing genres are only hints and may be inaccurate.
3) Select labels ONLY from the allowed lists.
4) NEVER invent new labels, synonyms, or variations.
5) Labels must match EXACT spelling including underscores.
6) Be conservative. Assign labels ONLY when clearly dominant.
7) Prefer fewer strong labels over many weak ones.
8) If unsure between assigning a label or none, choose none.

Avoid assigning broad philosophical themes (identity, morality, existentialism)
unless explicitly central to the narrative.

OUTPUT FORMAT (STRICT):

Tone: label1,label2 OR none
Pacing: label1,label2 OR none
Aesthetic: label1,label2 OR none
Themes: label1,label2,label3 OR none

Formatting rules:
- No extra text.
- No explanations.
- No analysis.
- Use lowercase exactly as defined.
- Separate multiple labels using commas with NO spaces.

Allowed Tone Labels:
dark, melancholic, hopeful, tragic, uplifting, introspective, tense, humorous, romantic, mysterious, epic, whimsical

Allowed Pacing Labels:
slow_burn, fast_paced, character_driven, plot_driven, episodic

Allowed Aesthetic Labels:
dark_academia, cyberpunk, gothic, philosophical, cozy, surreal, dystopian, mythological, historical, high_fantasy, urban_fantasy, literary, science_fiction

Allowed Theme Labels:
coming_of_age, revenge, redemption, war, politics, technology, family, survival, love, loss, identity, existentialism, morality
INPUT:

Title:
{title}

Existing Genres (may be noisy):
{genres}

Book Summary:
{summary}
"""

In [24]:
import re
def parse_labels(llm_output):
    #Initalize empty dictionary
    data = {"Tone": "", "Pacing": "", "Aesthetic": "", "Themes": ""}
    patterns = {
        "Tone": r"Tone:\s*(.*)",
        "Pacing": r"Pacing:\s*(.*)",
        "Aesthetic": r"Aesthetic:\s*(.*)",
        "Themes": r"Themes:\s*(.*)"
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, llm_output)
        if match:
            data[key] = match.group(1).strip()
    return data

In [26]:
import time
test_chunk = df.sample(n=200, random_state=42)
count = 1
for index, row in test_chunk.iterrows():
    start_time = time.time()

    user_input = f"""
Title:
{row['Title']}

Existing Genres (may be noisy):
{row['Genres']}

Book Summary:
{row['Summary']}
"""

    chat_completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.3,
        messages=[
            {"role": "system", "content": promptCh},
            {"role": "user", "content": user_input}
        ]
    )

    llm_output = chat_completion.choices[0].message.content

    parsed_data = parse_labels(llm_output)

    for category, labels in parsed_data.items():
        test_chunk.at[index, category] = labels

    print(f"[{count}/200] Index: {index} Classified: {row['Title']}")
    count += 1
    elapsed = time.time()
    if(elapsed < 3):
        time.sleep(3-elapsed)

[1/200] Index: 3169 Classified: The Boy Who Lost His Face
[2/200] Index: 4257 Classified: Tunnels of Blood
[3/200] Index: 4130 Classified: The Franchise
[4/200] Index: 2715 Classified: The Lady in the Lake
[5/200] Index: 10284 Classified: Skinnybones
[6/200] Index: 6292 Classified: The Family Trade
[7/200] Index: 11941 Classified: The Sand Dwellers
[8/200] Index: 2301 Classified: The Clerk's Prologue and Tale
[9/200] Index: 2718 Classified: The House of the Scorpion
[10/200] Index: 12297 Classified: The 3 Mistakes of My Life
[11/200] Index: 4104 Classified: Flyte
[12/200] Index: 9300 Classified: No Dominion
[13/200] Index: 6760 Classified: Haunted
[14/200] Index: 6345 Classified: Ordinary Jack
[15/200] Index: 16059 Classified: Melmoth
[16/200] Index: 5561 Classified: The Tale of Mr. Jeremy Fisher
[17/200] Index: 2271 Classified: Dream Story
[18/200] Index: 8937 Classified: The Measure of a Man: A Spiritual Autobiography
[19/200] Index: 426 Classified: War and Peace
[20/200] Index: 3325

In [35]:
test_chunk.sample(20)

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes
6822,Gil's All Fright Diner,A. Lee Martinez,"['Science Fiction', 'Speculative fiction']","In the backwoods southern town of Rockwood, a...",humorous,fast_paced,none,none
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,"introspective, melancholic",slow_burn,philosophical,"existentialism, identity, morality, power_corr..."
8990,The Case of Miss Elliot,Baroness Emma Orczy,['Uncategorized'],"Despite his vanity about his own talents, Bil...",humorous,character_driven,none,"identity, morality"
8147,Thirteen Moons,Charles Frazier,['Historical novel'],"Near the end of his life, frontiersman Will C...","melancholic,tragic",slow_burn,historical,"identity,loss,power_corruption,love,survival"
16059,Melmoth,NaN,['Uncategorized'],"After Jaka, Rick and Oscar's arrest (and Pud'...",,,,
4794,How I Survived My Summer Vacation,NaN,['Horror'],:Written by Michelle Sagara West Buffy contin...,"hopeful, melancholic",fast_paced,none,"identity, loss, redemption"
13123,The Wicked Witch of Oz,Rachel Cosgrove Payes,['Fantasy'],"The title character is Singra, the Wicked Wit...","hopeful, tragic",fast_paced,none,"redemption, morality"
11415,2nd Chance,James Patterson,"['Crime Fiction', 'Mystery', 'Fiction', 'Suspe...",Homicide Lieutenant Lindsay Boxer is still re...,"tragic, hopeful",fast_paced,none,"revenge, identity"
5701,The House of the Seven Gables,Nathaniel Hawthorne,"['Horror', 'Mystery', 'American Gothic Fiction...","The novel is set in the mid-19th century, wit...","melancholic,tragic",character_driven,gothic,"identity, power_corruption, redemption, loss"
10949,Paloma,Kristine Kathryn Rusch,"['Science Fiction', 'Speculative fiction', 'De...","Miles Flint, a retrieval artist, returns to t...",,,,


In [36]:
theme_series = (
    test_chunk['Themes']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print(theme_series.value_counts())

Themes
identity            137
power_corruption    101
morality             61
loss                 54
survival             44
redemption           42
love                 36
family               35
existentialism       23
                     22
coming_of_age        17
revenge              14
war                  12
none                 10
society              10
technology            7
politics              2
friendship            2
innocence             1
guilt                 1
judgement             1
apocalypse            1
Name: count, dtype: int64


In [37]:
test_chunk['meaning'] = (
    "Title: " + test_chunk['Title'].fillna('') +
    " Summary: " + test_chunk['Summary'].fillna('') +
    " Tone: " + test_chunk['Tone'].fillna('none') +
    " Pacing: " + test_chunk['Pacing'].fillna('none') +
    " Aesthetic: " + test_chunk['Aesthetic'].fillna('none') +
    " Themes: " + test_chunk['Themes'].fillna('none')
)

In [38]:
test_chunk.head()

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes,meaning
3169,The Boy Who Lost His Face,Louis Sachar,"[""Children's literature"", 'Young adult literat...","In a 1989 suburban town, a boy named David tr...","hopeful, introspective, humorous",character_driven,none,"identity, coming_of_age, redemption",Title: The Boy Who Lost His Face Summary: In ...
4257,Tunnels of Blood,Darren Shan,"['Young adult literature', ""Children's literat...","This story introduces Gavner Purl, a full vam...","hopeful, romantic",fast_paced,none,"identity, love, survival",Title: Tunnels of Blood Summary: This story i...
4130,The Franchise,Peter Gent,['Uncategorized'],Taylor Rusk is a star college quarterback and...,"dark,tragic",fast_paced,none,"power_corruption, identity, revenge",Title: The Franchise Summary: Taylor Rusk is ...
2715,The Lady in the Lake,Raymond Chandler,"['Crime Fiction', 'Detective fiction', 'Novel'...","Derace Kingsley, a wealthy businessman, hires...","dark, tragic",slow_burn,none,"power_corruption, revenge, identity, morality",Title: The Lady in the Lake Summary: Derace K...
10284,Skinnybones,Barbara Park,"[""Children's literature"", 'Fiction']",Alex wrote a letter in a promotional contest ...,"hopeful, humorous",fast_paced,none,"identity, coming_of_age",Title: Skinnybones Summary: Alex wrote a lett...


In [39]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

/home/shivam/Projects/Books4U/server-model/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 586.41it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
print("Generating embeddings .....")
embeddings = model.encode(test_chunk['meaning'].to_list(), show_progress_bar=True)

print("Embeddings shape ", embeddings.shape)

Generating embeddings .....


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:02<00:00,  2.64it/s]

Embeddings shape  (200, 384)


In [41]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
def search_books(query, top_n=5):
    query_vector = model.encode([query])
    similarities = cosine_similarity(query_vector, embeddings)[0]
    top_indices = np.argsort(similarities)[-top_n:][::-1]
    results = test_chunk.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    
    return results[['Title', 'Author', 'Genres', 'similarity_score']]

In [42]:
search_books("dark academia")

,Title,Author,Genres,similarity_score
5157,Nemesis,Scott Ciencin,"['Speculative fiction', 'Horror']",0.404754
13243,Blackthorn Winter,Kathryn Reiss,"['Mystery', 'Young adult literature']",0.373075
16118,Move Under Ground,Nick Mamatas,"['Horror', 'Speculative fiction', 'Mystery', '...",0.335953
9300,No Dominion,Charlie Huston,"['Thriller', 'Horror', 'Speculative fiction', ...",0.328588
8046,None But Lucifer,H. L. Gold,"['Speculative fiction', 'Fantasy']",0.297866


In [43]:
search_books("character-driven philosophical sci-fi")

,Title,Author,Genres,similarity_score
16421,Destiny Times Three,Fritz Leiber,"['Science Fiction', 'Novel']",0.371650
9781,Est: The Steersman Handbook,Leslie Stevens,['Science Fiction'],0.354842
7765,Salt,Adam Roberts,"['Science Fiction', 'Speculative fiction']",0.343208
5157,Nemesis,Scott Ciencin,"['Speculative fiction', 'Horror']",0.339165
11864,Operation: Outer Space,Murray Leinster,"['Science Fiction', 'Speculative fiction']",0.333407


In [44]:
search_books("political dystopia.")

,Title,Author,Genres,similarity_score
16421,Destiny Times Three,Fritz Leiber,"['Science Fiction', 'Novel']",0.297160
4993,This Other Eden,Ben Elton,"['Speculative fiction', 'Fiction', 'Dystopia']",0.293488
7765,Salt,Adam Roberts,"['Science Fiction', 'Speculative fiction']",0.281931
9781,Est: The Steersman Handbook,Leslie Stevens,['Science Fiction'],0.259705
5681,Icon,Frederick Forsyth,"['Thriller', 'Novel']",0.253788


In [45]:
search_books("time loop story and philosophy")

,Title,Author,Genres,similarity_score
14251,Lowell Park,NaN,"['Historical fiction', 'Historical novel']",0.456626
16421,Destiny Times Three,Fritz Leiber,"['Science Fiction', 'Novel']",0.402985
291,Picnic at Hanging Rock,Joan Lindsay,"['Mystery', 'Historical novel']",0.369295
3826,The Time of the Ghost,Diana Wynne Jones,"['Ghost story', ""Children's literature"", 'Spec...",0.352429
8816,In Times Like These,Zee Edgell,"['History', 'Novel']",0.350784


In [46]:
search_books("moral dilemma involving artificial intelligence")

,Title,Author,Genres,similarity_score
4993,This Other Eden,Ben Elton,"['Speculative fiction', 'Fiction', 'Dystopia']",0.268198
9272,I Am the Messenger,Markus Zusak,['Fiction'],0.254824
2140,The Stupidest Angel,Christopher Moore,"['Speculative fiction', 'Horror', 'Comic fanta...",0.247489
8990,The Case of Miss Elliot,Baroness Emma Orczy,['Uncategorized'],0.227795
12545,Designing Economic Mechanisms,Leonid Hurwicz,['Uncategorized'],0.227704


In [47]:
search_books("lonely but beautiful")

,Title,Author,Genres,similarity_score
14376,Impossible,Danielle Steel,['Novel'],0.319853
15705,Luv Ya Bunches,Lauren Myracle,['Literary fiction'],0.306727
16174,Busabos ng Palad,NaN,['Novel'],0.278655
5532,Acorna's People,Elizabeth Ann Scarborough,"['Science Fiction', 'Speculative fiction', 'Fa...",0.260821
6081,The Hunger of Sejanoz,Joe Dever,"['Gamebook', ""Children's literature""]",0.256551
